In [1]:
import datacube
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import os

import glob
import imageio
import numpy as np
from PIL import Image

from odc.ui import with_ui_cbk
from datacube.utils.cog import write_cog

import sys
sys.path.insert(1, "../Tools/")
from dea_tools.plotting import rgb, display_map
from dea_tools.landcover import plot_land_cover

from matplotlib.colors import ListedColormap
from matplotlib import colors as mcolours

In [2]:
def get_product_for_year_3(year):
    """
    Get the product name based on the given year.

    Parameters:
    - year (int): year for which product is needed.

    Returns:
    - product (str): product name.
    """
    if 1988 <= year <= 1999:
        return "ga_ls5t_nbart_gm_cyear_3"
    elif 2000 <= year <= 2003:
        return "ga_ls7e_nbart_gm_cyear_3"
    elif 2004 <= year <= 2007:
        return "ga_ls5t_nbart_gm_cyear_3"
    elif year == 2008:
        return "ga_ls7e_nbart_gm_cyear_3"
    elif 2009 <= year <= 2011:
        return "ga_ls5t_nbart_gm_cyear_3"
    elif year == 2012:
        return "ga_ls7e_nbart_gm_cyear_3"
    elif year >= 2013:
        return "ga_ls8c_nbart_gm_cyear_3"
    else:
        return None

def get_product_for_year_4(year):
    """
    Get the product name based on the given year.

    Parameters:
    - year (int): year for which product is needed.

    Returns:
    - product (str): product name.
    """
    if 1988 <= year <= 1999:
        return "ga_ls5t_gm_cyear_3"
    elif 2000 <= year <= 2003:
        return "ga_ls7e_gm_cyear_3"
    elif 2004 <= year <= 2007:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2008:
        return "ga_ls7e_gm_cyear_3"
    elif 2009 <= year <= 2011:
        return "ga_ls5t_gm_cyear_3"
    elif year == 2012:
        return "ga_ls7e_gm_cyear_3"
    elif year >= 2013:
        return "ga_ls8cls9c_gm_cyear_3"
    else:
        return None
    
def load_geomedian_data(product, measurements, lon_range, lat_range, year):
    """
    Load geomedian dataset for a given year.

    Parameters:
    - product (str): product name.
    - measurements (list): list of measurement names.
    - lon_range (tuple): longitude range.
    - lat_range (tuple): latitude range.
    - year (int): year for which data is to be loaded.

    Returns:
    - geomedian (xarray.Dataset): Loaded geomedian dataset.
    """
    dc = datacube.Datacube(app="")
    geomedian = dc.load(
        product=product,
        measurements=measurements,
        x=lon_range,
        y=lat_range,
        time=(str(year), str(year)),
    )
    return geomedian

In [3]:
def make_gif(frame_folder, start_year, end_year, location):
    """
    Create a gif from a folder of images (png). 
    Parameters:
    - frame folder (str): path to the folder containing the images. 
    - start_year (int): starting year for the GIF title.
    - end_year (int): ending year for the GIF title.
    - location (str): location for the GIF title.
    Returns:
    - the gif will be saved in the current folder. 
    """
    frames = [Image.open(image) for image in sorted(glob.glob(f"{frame_folder}/*.png"))]
    frame_one = frames[0]
    frame_one.save(f"Geomedian 3v4 {start_year}-{end_year} {location}.gif", format="GIF", append_images=frames,
               save_all=True, duration=500, loop=0, optimize=True)

In [4]:
def delete_generated_images(folder):
    """
    Delete all the images from the 'images' directory.

    Parameters:
    - folder (str): name of folder/folder path

    Returns:
    - None
    """
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")

In [25]:
# location = "Bauxite Mine, WA"
# lat = -32.5549
# lon = 116.1667
# lat_buffer = 0.2
# lon_buffer = 0.3

# location = "North Brisbane, Qld"
# lat = -27.2731
# lon = 152.9970
# lat_buffer = 0.1
# lon_buffer = 0.15

# location = "Canberra, ACT"
# lat = -35.3062
# lon =  149.1216
# lat_buffer = 0.2
# lon_buffer = 0.3

# location = "Hobart, Tas"
# lat = -42.8820
# lon = 147.3328
# lat_buffer = 0.2
# lon_buffer = 0.4

location = "Devonport, Tas"
lat = -41.1758
lon = 146.3605
lat_buffer = 0.1
lon_buffer = 0.2


location = "Sydney, NSW"
lat = -33.8841
lon = 151.0517
lat_buffer = 0.1
lon_buffer = 0.2

lat_range = (lat - lat_buffer, lat + lat_buffer)
lon_range = (lon - lon_buffer, lon + lon_buffer)

In [26]:
display_map(x=lon_range, y=lat_range)

In [27]:
def make_plots(start_year, end_year):
    for year in range(start_year, end_year + 1):
        
        # load geomedian v3
        gm_bands = ["red", "green", "blue"]
        product3 = get_product_for_year_3(year)
        geomedian3 = load_geomedian_data(product3, gm_bands, lon_range, lat_range, year)
        
        # load geomedian v4
        gm_bands = ["nbart_red", "nbart_green", "nbart_blue"]
        product4 = get_product_for_year_4(year)
        geomedian4 = load_geomedian_data(product4, gm_bands, lon_range, lat_range, year)
        f, axarr = plt.subplots(1, 2, figsize=(15, 6), squeeze=False)
        
        
        
        rgb(
            geomedian3,
            bands=["red", "green", "blue"],
            ax=axarr[0, 0],
            robust=True,
        )
        axarr[0, 0].set_title("DEA Geomedian Version 3")  
        axarr[0, 0].text(0.97, 0.97, year, transform=axarr[0, 0].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))
        
        
        rgb(
            geomedian4,
            bands=["nbart_red", "nbart_green", "nbart_blue"],
            ax=axarr[0, 1],
            robust=True,
        )
        axarr[0, 1].set_title("DEA Geomedian Version 4")
        axarr[0, 1].text(0.97, 0.97, year, transform=axarr[0, 1].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))
        
        # remove uggo text and stuff 
        axarr[0, 0].set_xlabel('')
        axarr[0, 0].set_ylabel('')
        axarr[0, 0].set_xticks([])
        axarr[0, 0].set_yticks([])
        axarr[0, 1].set_xlabel('')
        axarr[0, 1].set_ylabel('')
        axarr[0, 1].set_xticks([])
        axarr[0, 1].set_yticks([])

        plt.tight_layout()  

        plt.savefig(f'images/{year}_{location}.png', dpi=150)

        # plt.show()
        plt.close()


In [28]:
make_plots(1988, 2020)

In [29]:
make_gif("images", 1988, 2020, location)

In [30]:
delete_generated_images("images")

In [78]:
lc_bands = ["level4"]
landcover = load_landcover_data(lc_bands, lon_range, lat_range, 2015)
# lc

In [79]:
gm_bands = ["nbart_red", "nbart_green", "nbart_blue"]
product = get_product_for_year(year)
geomedian = load_geomedian_data(product, gm_bands, lon_range, lat_range, year)
# geomedian

In [ ]:
f, axarr = plt.subplots(1, 2, figsize=(12, 6), squeeze=False)

rgb(
    geomedian,
    bands=["nbart_red", "nbart_green", "nbart_blue"],
    ax=axarr[0, 0],
    robust=True,
)
# axarr[0, 0].set_title("DEA Geomedian")  
# axarr[0, 0].set_title("")  
axarr[0, 0].text(0.97, 0.97, year, transform=axarr[0, 0].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))

plot_layer(level4_cmap, landcover.level4, ax=axarr[0, 1]) 
# axarr[0, 1].set_title("DEA Land Cover")
axarr[0, 1].set_title("")
axarr[0, 1].text(0.97, 0.97, year, transform=axarr[0, 1].transAxes, fontsize=10, ha='right', va='top', color='white', bbox=dict(facecolor='black', edgecolor='black'))


axarr[0, 0].set_xlabel('')
axarr[0, 0].set_ylabel('')
axarr[0, 0].set_xticks([])
axarr[0, 0].set_yticks([])
axarr[0, 1].set_xlabel('')
axarr[0, 1].set_ylabel('')
axarr[0, 1].set_xticks([])
axarr[0, 1].set_yticks([])

plt.tight_layout()  

plt.savefig(f'images/{year}_{location}.png', dpi=150)

plt.show()